# Inference OFA — Multi-Character Aksara Lontara
## Model: Fine-tuned `OFA-Sys/ofa-base`

Notebook ini melakukan inference menggunakan model **OFA** yang sudah di-fine-tune.
Mendukung deteksi **1 sampai 4 karakter** per gambar.

**Alur kerja:**
1. Preprocessing → Grayscale → Adaptive threshold
2. Vertical Projection Segmentation → Split karakter
3. Crop + Resize (stretch) 384×384 — konsisten dengan `training_ofa.ipynb`
4. Inference individual per karakter (prompt tetap + patch_images OFA)
5. Penggabungan caption

> **Catatan konsistensi:** preprocessing & `MAX_LENGTH=128` disamakan dengan `training_ofa.ipynb` (sumber `model/ofa`), yang me-resize gambar penuh ke 384×384 tanpa padding.

> **PENTING (OFA):** sel Setup meng-install **fork** `OFA-Sys/OFA` (branch `feature/add_transformers`) yang menyediakan `OFAModel` & `OFATokenizer`.

---
## Setup & Load Model

In [ ]:
# Install fork transformers OFA (menyediakan OFAModel & OFATokenizer)
import os
if not os.path.exists("/content/OFA"):
    !git clone --single-branch --branch feature/add_transformers https://github.com/OFA-Sys/OFA.git /content/OFA
!pip install -q /content/OFA/transformers/
!pip install -q Pillow

import re
import json
import cv2
import math
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from transformers import OFATokenizer, OFAModel
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR  = "/content/drive/MyDrive/ImageCaptioning"
MODEL_DIR  = f"{DRIVE_DIR}/model/ofa/best_model"
TEST_DIR   = f"{DRIVE_DIR}/datatest"
MAX_LENGTH = 128
NUM_BEAMS  = 4
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_SHORT = "ofa"
PROMPT     = " what does the image describe?"

IMAGE_SIZE = 384
OFA_MEAN = [0.5, 0.5, 0.5]
OFA_STD = [0.5, 0.5, 0.5]
patch_resize_transform = transforms.Compose([
    lambda image: image.convert("RGB"),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=OFA_MEAN, std=OFA_STD),
])

print(f"Device: {DEVICE}")
print(f"Model : {MODEL_DIR}")
print(f"Test  : {TEST_DIR}")

assert os.path.exists(MODEL_DIR), f"Model tidak ditemukan: {MODEL_DIR}"

tokenizer = OFATokenizer.from_pretrained(MODEL_DIR)
model = OFAModel.from_pretrained(MODEL_DIR, use_cache=False).to(DEVICE)
model.eval()
print("Model OFA siap!")

---
## Fungsi Segmentasi & Inference

In [ ]:
import re
import cv2
import math

MAX_CHARS    = 4
CANVAS_SIZE  = 384
BBOX_PAD_RATIO = 0.20


def preprocess_binary(image_path):
    """Konversi gambar ke binary mask (karakter = putih, bg = hitam)."""
    img_cv = cv2.imread(image_path)
    if img_cv is None:
        raise ValueError(f"Gagal membaca gambar: {image_path}")
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    thresh = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, blockSize=25, C=10
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=1)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
    return img_cv, gray, thresh


def find_split_points(projection, min_gap_width=3, valley_thresh=0):
    """Cari gap (kolom berturut-turut dengan projection <= valley_thresh)."""
    gaps = []
    in_gap = False
    gap_start = 0
    for i, val in enumerate(projection):
        if val <= valley_thresh:
            if not in_gap:
                gap_start = i
                in_gap = True
        else:
            if in_gap:
                gap_width = i - gap_start
                if gap_width >= min_gap_width:
                    gaps.append((gap_start, i, gap_width))
                in_gap = False
    if in_gap:
        gap_width = len(projection) - gap_start
        if gap_width >= min_gap_width:
            gaps.append((gap_start, len(projection), gap_width))
    return gaps


def segment_by_projection(image_path, min_gap_ratio=0.02, debug=False):
    """Segmentasi karakter menggunakan Vertical Projection Profile."""
    img_cv, gray, thresh = preprocess_binary(image_path)
    h_img, w_img = gray.shape
    projection = np.sum(thresh > 0, axis=0)
    min_gap_px = max(2, int(w_img * min_gap_ratio))
    max_proj = int(projection.max()) if len(projection) > 0 else 0
    valley_thresh = max(0, int(max_proj * 0.05))
    gaps = find_split_points(projection, min_gap_width=min_gap_px, valley_thresh=valley_thresh)

    if debug:
        print(f"  Image size: {w_img}x{h_img}")
        print(f"  Min gap: {min_gap_px}px, Valley thresh: {valley_thresh}")
        print(f"  Gaps found: {len(gaps)}")

    content_cols = np.where(projection > valley_thresh)[0]
    if len(content_cols) == 0:
        return [], img_cv, thresh, projection

    col_start = content_cols[0]
    col_end = content_cols[-1] + 1
    inner_gaps = [(gs, ge, gw) for gs, ge, gw in gaps if gs > col_start and ge < col_end]

    if len(inner_gaps) > MAX_CHARS - 1:
        inner_gaps = sorted(inner_gaps, key=lambda g: g[2], reverse=True)[:MAX_CHARS - 1]
        inner_gaps = sorted(inner_gaps, key=lambda g: g[0])

    split_cols = [col_start]
    for gs, ge, gw in inner_gaps:
        gap_segment = projection[gs:ge]
        min_idx = int(np.argmin(gap_segment))
        split_cols.append(gs + min_idx)
    split_cols.append(col_end)

    raw_boxes = []
    num_regions = len(split_cols) - 1
    for r in range(num_regions):
        x_left = split_cols[r]
        x_right = split_cols[r + 1]
        region_strip = thresh[:, x_left:x_right]
        row_proj = np.sum(region_strip > 0, axis=1)
        active_rows = np.where(row_proj > 0)[0]
        if len(active_rows) == 0:
            continue
        y_top = active_rows[0]
        y_bot = active_rows[-1] + 1
        region_w = x_right - x_left
        region_h = y_bot - y_top
        pad_x = max(5, int(region_w * 0.20))
        pad_y = max(5, int(region_h * 0.20))
        if r > 0:
            safe_x_left = inner_gaps[r - 1][0]
        else:
            safe_x_left = 0
        if r < num_regions - 1:
            safe_x_right = inner_gaps[r][1]
        else:
            safe_x_right = w_img
        x1 = max(safe_x_left, x_left - pad_x)
        y1 = max(0, y_top - pad_y)
        x2 = min(safe_x_right, x_right + pad_x)
        y2 = min(h_img, y_bot + pad_y)
        if (x2 - x1) > 5 and (y2 - y1) > 5:
            raw_boxes.append((x1, y1, x2, y2))

    if debug:
        print(f"  Karakter terdeteksi: {len(raw_boxes)}")

    return raw_boxes, img_cv, thresh, projection


def crop_character_on_canvas(pil_img, box, canvas_size=CANVAS_SIZE):
    """Crop karakter lalu stretch ke canvas_size x canvas_size.

    Konsisten dengan training_ofa.ipynb yang me-resize setiap gambar
    (stretch) ke 384x384 tanpa padding. Karakter mengisi seluruh frame,
    sama seperti distribusi data training.
    """
    x1, y1, x2, y2 = box
    cropped = pil_img.crop((x1, y1, x2, y2))
    return cropped.resize((canvas_size, canvas_size), Image.LANCZOS)


def detect_bbox_single(pil_image, pad_ratio=BBOX_PAD_RATIO):
    """Deteksi bbox tunggal untuk gambar 1 karakter (mirror training_ofa)."""
    img_np = np.array(pil_image.convert("RGB"))
    img_cv = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    thresh = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, blockSize=25, C=10
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=1)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
    h, w = thresh.shape
    cols = np.where(np.any(thresh > 0, axis=0))[0]
    rows = np.where(np.any(thresh > 0, axis=1))[0]
    if len(cols) == 0 or len(rows) == 0:
        return (0, 0, w, h)
    x1, x2 = int(cols[0]), int(cols[-1]) + 1
    y1, y2 = int(rows[0]), int(rows[-1]) + 1
    bw, bh = x2 - x1, y2 - y1
    pad_x = max(5, int(bw * pad_ratio))
    pad_y = max(5, int(bh * pad_ratio))
    return (max(0, x1 - pad_x), max(0, y1 - pad_y),
            min(w, x2 + pad_x), min(h, y2 + pad_y))


def extract_char_name(caption):
    """Ekstrak nama karakter dari caption model."""
    patterns = [
        r"yaitu karakter\s+(\w+)",
        r"adalah\s+(\w+)\s+dengan",
        r"merupakan karakter\s+(\w+)",
        r"karakter\s+(\w+)\s+memiliki",
        r"karakter\s+(\w+)\s+dengan",
    ]
    for pat in patterns:
        m = re.search(pat, caption, re.IGNORECASE)
        if m:
            return m.group(1)
    return None


def combine_captions(captions):
    """Gabungkan caption individual (1-4 karakter)."""
    n = len(captions)
    if n == 0:
        return "Tidak ada karakter yang terdeteksi."
    if n == 1:
        return captions[0]

    names = [extract_char_name(c) for c in captions]
    valid_names = [nm for nm in names if nm is not None]

    if len(valid_names) == n:
        if n == 2:
            nama_str = f"{valid_names[0]} dan {valid_names[1]}"
        else:
            nama_str = ", ".join(valid_names[:-1]) + f", dan {valid_names[-1]}"
        intro = f"Gambar ini terdiri dari {n} karakter lontara yaitu karakter {nama_str}."
    else:
        intro = f"Gambar ini terdiri dari {n} karakter lontara."

    details = []
    for i, cap in enumerate(captions, 1):
        name = names[i-1] if names[i-1] else "?"
        clean = re.sub(r"^Gambar ini terdiri dari \d+ karakter lontara[^.]*\.?\s*", "", cap).strip()
        if not clean:
            clean = cap
        details.append(f"Karakter ke-{i} ({name}): {clean}")

    return intro + " " + " ".join(details)


def predict_single(pil_img):
    """Captioning 1 gambar dengan OFA (prompt tetap + patch_images)."""
    patch_image = patch_resize_transform(pil_img).unsqueeze(0).to(DEVICE)
    prompt_ids = tokenizer([PROMPT], return_tensors="pt").input_ids.to(DEVICE)
    with torch.no_grad():
        gen = model.generate(
            prompt_ids,
            patch_images=patch_image,
            max_length=MAX_LENGTH,
            min_length=5,
            num_beams=NUM_BEAMS,
            early_stopping=True,
            no_repeat_ngram_size=2,
        )
    return tokenizer.batch_decode(gen, skip_special_tokens=True)[0].strip()


def predict_multi(image_path, debug=False):
    """Segmentasi -> crop (stretch 384) -> inference -> gabung.

    Jika hanya 1 karakter terdeteksi, gambar tetap di-crop ke bounding-box
    karakter lalu resize 384 (konsisten dgn training_ofa yang memakai
    bbox crop). Mencegah out-of-distribution full-frame.
    """
    pil_img = Image.open(image_path).convert("RGB")
    boxes, img_cv, thresh, projection = segment_by_projection(image_path, min_gap_ratio=0.02, debug=debug)
    if len(boxes) <= 1:
        # 1 karakter: pakai bbox tunggal supaya konsisten dengan training
        single_box = detect_bbox_single(pil_img)
        single_img = pil_img.crop(single_box).resize((CANVAS_SIZE, CANVAS_SIZE), Image.LANCZOS)
        cap = predict_single(single_img)
        return cap, [single_img], [cap], [single_box], projection
    char_imgs = []
    char_caps = []
    for box in boxes:
        cimg = crop_character_on_canvas(pil_img, box)
        char_imgs.append(cimg)
        char_caps.append(predict_single(cimg))
    combined = combine_captions(char_caps)
    return combined, char_imgs, char_caps, boxes, projection

print(f"Segmentasi + Resize {CANVAS_SIZE}x{CANVAS_SIZE} (stretch + bbox single-char, konsisten training) siap!")

---
## Demo Inference (Satu Gambar)

In [ ]:
TEST_IMAGE = f"{TEST_DIR}/contoh_multi.png"   # <-- sesuaikan nama file

if not os.path.exists(TEST_IMAGE):
    print(f"Gambar tidak ditemukan: {TEST_IMAGE}")
    print("Sesuaikan variabel TEST_IMAGE dengan nama file Anda.")
else:
    combined, char_imgs, char_caps, boxes, projection = predict_multi(TEST_IMAGE, debug=True)
    n = len(char_imgs)
# ─── Visualisasi Debug ───
img_cv_dbg, _, thresh_dbg = preprocess_binary(TEST_IMAGE)
h_img, w_img = thresh_dbg.shape

region_colors = [(255, 80, 80), (80, 200, 80), (80, 120, 255), (255, 180, 50)]

colored_mask = np.ones((h_img, w_img, 3), dtype=np.uint8) * 30
if len(boxes) > 0:
    for idx_b, (x1, y1, x2, y2) in enumerate(boxes):
        clr = region_colors[idx_b % len(region_colors)]
        region = thresh_dbg[y1:y2, x1:x2]
        for c in range(3):
            colored_mask[y1:y2, x1:x2, c] = np.where(
                region > 0, clr[c], colored_mask[y1:y2, x1:x2, c])
        cv2.rectangle(colored_mask, (x1, y1), (x2, y2), (255, 255, 255), 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

img_asli_draw = cv2.cvtColor(img_cv_dbg, cv2.COLOR_BGR2RGB).copy()
for idx_b, (x1, y1, x2, y2) in enumerate(boxes):
    clr = region_colors[idx_b % len(region_colors)]
    cv2.rectangle(img_asli_draw, (x1, y1), (x2, y2), clr, 2)
    cv2.putText(img_asli_draw, f"Chr {idx_b+1}", (x1, max(y1-5, 12)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, clr, 1)
axes[0].imshow(img_asli_draw)
axes[0].set_title("Gambar Asli + Bounding Box", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(colored_mask)
axes[1].set_title("Region per Karakter", fontweight="bold")
axes[1].axis("off")

axes[2].fill_between(range(len(projection)), projection, alpha=0.4, color="gray")
axes[2].set_title("Vertical Projection", fontweight="bold")
axes[2].set_xlabel("Kolom (x)")
axes[2].set_ylabel("Piksel")
for idx_b, (x1, y1, x2, y2) in enumerate(boxes):
    clr_norm = tuple(c / 255.0 for c in region_colors[idx_b % len(region_colors)])
    axes[2].fill_between(range(x1, x2), projection[x1:x2], alpha=0.5, color=clr_norm)

plt.suptitle(f"Debug Segmentasi - {n} karakter terdeteksi", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ─── Visualisasi Hasil ───
if n >= 1:
    fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4))
    if n + 1 == 1:
        axes = [axes]
    img_draw = np.array(Image.open(TEST_IMAGE).convert("RGB")).copy()
    for idx_b, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        clr = region_colors[idx_b % len(region_colors)]
        cv2.rectangle(img_draw, (x1, y1), (x2, y2), clr, 3)
    axes[0].imshow(img_draw)
    axes[0].set_title("Input", fontweight="bold")
    axes[0].axis("off")
    for i in range(n):
        axes[i+1].imshow(char_imgs[i])
        axes[i+1].set_title(f"Karakter {i+1}", fontweight="bold")
        axes[i+1].axis("off")
    plt.tight_layout()
    plt.show()

print("\n" + "=" * 60)
print("CAPTION GABUNGAN:")
print(combined)
print("=" * 60)
print("DETAIL PER KARAKTER:")
for i, cap in enumerate(char_caps, 1):
    print(f"  [{i}] {cap}")

---
## Batch Inference (Seluruh Folder)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# BATCH INFERENCE (SELURUH FOLDER DATATEST)
# ═══════════════════════════════════════════════════════════════

VALID_EXT = (".png", ".jpg", ".jpeg")

if not os.path.exists(TEST_DIR):
    print(f"Folder {TEST_DIR} tidak ditemukan.")
else:
    images = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith(VALID_EXT)])
    print(f"Ditemukan {len(images)} gambar di folder datatest\n")

    results = []
    for img_name in images:
        img_path = os.path.join(TEST_DIR, img_name)
        try:
            cap, char_imgs_b, char_caps_b, boxes_b, _ = predict_multi(img_path)
            n_chars = len(char_imgs_b)
            results.append({
                "gambar": img_name,
                "jumlah_karakter": n_chars,
                "caption_gabungan": cap,
                "caption_per_karakter": char_caps_b
            })
            print(f"{img_name} ({n_chars} char) -> {cap}")
        except Exception as e:
            print(f"{img_name} -> ERROR: {e}")

    out_dir = f"{DRIVE_DIR}/hasil_inference_{MODEL_SHORT}"
    os.makedirs(out_dir, exist_ok=True)
    out_file = f"{out_dir}/hasil_multi.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"\nHasil disimpan: {out_file}")

    # Grid visualisasi
    show = results[:12]
    cols = min(4, len(show))
    rows = math.ceil(len(show) / cols) if cols > 0 else 1
    if len(show) > 0:
        fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
        if rows * cols == 1:
            axes = [axes]
        else:
            axes = np.array(axes).flatten()
        for idx, item in enumerate(show):
            img = Image.open(os.path.join(TEST_DIR, item["gambar"]))
            axes[idx].imshow(img)
            axes[idx].set_title(f"{item['gambar']}\n({item['jumlah_karakter']} char)", fontsize=8)
            axes[idx].axis("off")
        for idx in range(len(show), len(axes)):
            axes[idx].axis("off")
        plt.tight_layout()
        plt.show()

    print(f"\nSelesai! Total {len(results)} gambar diproses.")